# NB06 FIXED — Statistical Comparison, Calibration, and Publication Figures

This notebook aggregates NB03–NB05 results and performs pre-specified, leakage-safe statistical comparisons using aligned out-of-fold (OOF) predictions. It also computes calibration summaries and publication-ready figures. Pairwise tests are restricted to scientifically motivated comparisons rather than every possible pair, reducing multiplicity and runtime.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
RESULTS = ROOT/'03_RESULTS'
FIGURES = ROOT/'05_FIGURES'
TABLES = ROOT/'06_TABLES'
LOGS = ROOT/'07_LOGS'
DATA = ROOT/'01_DATA'
TARGET='Class'
SEEDS=[2026,2027,2028]

OUT=RESULTS/'NB06_STATS'; OUT.mkdir(parents=True,exist_ok=True)
TAB=TABLES/'NB06_STATS'; TAB.mkdir(parents=True,exist_ok=True)
FIG=FIGURES/'NB06_STATS'; FIG.mkdir(parents=True,exist_ok=True)
LOG=LOGS/'NB06_STATS'; LOG.mkdir(parents=True,exist_ok=True)

metric_files=[
    RESULTS/'NB03_BASELINES'/'baseline_metrics_by_fold.csv',
    RESULTS/'NB04_HYBRIDS'/'hybrid_metrics_by_fold.csv',
    RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_metrics_by_fold.csv',
]
pred_files=[
    RESULTS/'NB03_BASELINES'/'baseline_oof_predictions.csv',
    RESULTS/'NB04_HYBRIDS'/'hybrid_oof_predictions.csv',
    RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_oof_predictions.csv',
]
for p in metric_files+pred_files:
    assert p.exists(), f'Missing required file: {p}'

metrics=pd.concat([pd.read_csv(p) for p in metric_files],ignore_index=True)
preds=pd.concat([pd.read_csv(p) for p in pred_files],ignore_index=True)
metrics.to_csv(OUT/'all_model_metrics_by_fold.csv',index=False)
preds.to_csv(OUT/'all_oof_predictions.csv',index=False)
print('Models:', sorted(metrics.model.unique()))
print('Metric rows:', len(metrics), 'OOF rows:', len(preds))


In [ ]:

# Integrity checks
expected_models = {
    'LogisticRegression','SVM_RBF','XGBoost','MLP',
    'PCA_SVM','PCA_MLP','PCA_XGBoost',
    'LDA_SVM','LDA_MLP','LDA_XGBoost',
    'TabNet','FTTransformer'
}
assert set(metrics.model.unique()) == expected_models, (set(metrics.model.unique()), expected_models)
assert metrics.groupby('model').size().eq(15).all(), metrics.groupby('model').size()
assert preds.groupby('model').size().eq(12000).all(), preds.groupby('model').size()
assert preds.groupby(['model','seed']).row_id.nunique().eq(4000).all()
print('Integrity checks PASSED: 12 models, 15 outer-fold evaluations/model, 4000 OOF grains/model/seed.')


In [ ]:

# Descriptive comparison
summary=metrics.groupby('model').agg(
    mean_macro_f1=('f1_macro','mean'),
    sd_macro_f1=('f1_macro','std'),
    mean_accuracy=('accuracy','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_mcc=('mcc','mean'),
    mean_kappa=('kappa','mean'),
    mean_log_loss=('log_loss','mean'),
).sort_values('mean_macro_f1',ascending=False)
summary.to_csv(TAB/'all_models_summary.csv')
display(summary)

seed_summary=metrics.groupby(['model','seed']).agg(
    macro_f1=('f1_macro','mean'), accuracy=('accuracy','mean'),
    balanced_accuracy=('balanced_accuracy','mean'), mcc=('mcc','mean'),
    kappa=('kappa','mean'), log_loss=('log_loss','mean')
).reset_index()
seed_summary.to_csv(TAB/'all_models_summary_by_seed.csv',index=False)


In [ ]:

# Calibration summaries from OOF probabilities: multiclass Brier score and ECE (10 bins)
classes=['INIAP 420','INIAP 425','INIAP 481','INIAP 485']
prob_cols=[f'prob_{c}' for c in classes]

def multiclass_brier(y_true, probs, classes):
    y_idx=pd.Categorical(y_true,categories=classes).codes
    one=np.eye(len(classes))[y_idx]
    return np.mean(np.sum((probs-one)**2,axis=1))

def ece_score(y_true, y_pred, probs, n_bins=10):
    conf=probs.max(axis=1)
    correct=(np.asarray(y_true)==np.asarray(y_pred)).astype(float)
    bins=np.linspace(0,1,n_bins+1)
    ece=0.0
    for i in range(n_bins):
        lo,hi=bins[i],bins[i+1]
        m=(conf>=lo)&((conf<hi) if i<n_bins-1 else (conf<=hi))
        if m.any():
            ece += m.mean()*abs(correct[m].mean()-conf[m].mean())
    return float(ece)

cal_rows=[]
for (model,seed),g in preds.groupby(['model','seed']):
    if not set(prob_cols).issubset(g.columns):
        continue
    gg=g.drop_duplicates('row_id').sort_values('row_id')
    P=gg[prob_cols].to_numpy(float)
    # tolerate small numerical drift only
    P=np.clip(P,1e-12,1.0)
    P=P/P.sum(axis=1,keepdims=True)
    cal_rows.append({
        'model':model,'seed':int(seed),
        'brier_multiclass':multiclass_brier(gg.y_true.astype(str),P,classes),
        'ece_10bin':ece_score(gg.y_true.astype(str),gg.y_pred.astype(str),P,10),
    })
cal=pd.DataFrame(cal_rows)
cal.to_csv(TAB/'calibration_by_model_seed.csv',index=False)
cal_summary=cal.groupby('model').agg(
    mean_brier=('brier_multiclass','mean'), sd_brier=('brier_multiclass','std'),
    mean_ece=('ece_10bin','mean'), sd_ece=('ece_10bin','std')
).sort_values('mean_brier')
cal_summary.to_csv(TAB/'calibration_summary.csv')
display(cal_summary)


## Pre-specified primary comparisons

These comparisons directly test the scientific questions of the manuscript: whether supervised or unsupervised dimensionality reduction changes performance relative to the corresponding direct classifier, and how the strongest hybrid compares with advanced tabular deep-learning comparators.


In [ ]:

from sklearn.metrics import f1_score
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

PRIMARY_COMPARISONS=[
    ('LDA_XGBoost','XGBoost'),
    ('LDA_SVM','SVM_RBF'),
    ('LDA_MLP','MLP'),
    ('PCA_XGBoost','XGBoost'),
    ('PCA_SVM','SVM_RBF'),
    ('PCA_MLP','MLP'),
    ('LDA_XGBoost','FTTransformer'),
    ('LDA_XGBoost','TabNet'),
    ('LDA_XGBoost','LDA_SVM'),
]

rng=np.random.default_rng(2026)
B=3000
boot_rows=[]
mc_rows=[]

for seed in SEEDS:
    ps=preds[preds.seed==seed]
    for a,b in PRIMARY_COMPARISONS:
        A=ps[ps.model==a].drop_duplicates('row_id').set_index('row_id').sort_index()
        Bp=ps[ps.model==b].drop_duplicates('row_id').set_index('row_id').sort_index()
        idx=A.index.intersection(Bp.index)
        assert len(idx)==4000, (seed,a,b,len(idx))
        yt=A.loc[idx,'y_true'].astype(str).to_numpy()
        ya=A.loc[idx,'y_pred'].astype(str).to_numpy()
        yb=Bp.loc[idx,'y_pred'].astype(str).to_numpy()
        obs=f1_score(yt,ya,average='macro')-f1_score(yt,yb,average='macro')
        diffs=np.empty(B,dtype=float)
        n=len(idx)
        for i in range(B):
            s=rng.integers(0,n,n)
            diffs[i]=f1_score(yt[s],ya[s],average='macro')-f1_score(yt[s],yb[s],average='macro')
        lo,hi=np.quantile(diffs,[.025,.975])
        boot_rows.append({
            'seed':seed,'model_a':a,'model_b':b,
            'delta_macro_f1_a_minus_b':obs,'ci95_low':lo,'ci95_high':hi,
            'ci_excludes_zero':bool(lo>0 or hi<0),
            'ci_entirely_within_practical_margin_0.01':bool(lo>-0.01 and hi<0.01),
        })

        ca=(ya==yt); cb=(yb==yt)
        n10=int(np.sum(ca & ~cb)); n01=int(np.sum(~ca & cb)); ndisc=n10+n01
        p=1.0 if ndisc==0 else binomtest(n10,n=ndisc,p=.5,alternative='two-sided').pvalue
        mc_rows.append({'seed':seed,'model_a':a,'model_b':b,
                        'a_correct_b_wrong':n10,'a_wrong_b_correct':n01,
                        'discordant_pairs':ndisc,'p_exact':p})

boot=pd.DataFrame(boot_rows)
boot.to_csv(TAB/'paired_bootstrap_primary_macro_f1.csv',index=False)

mc=pd.DataFrame(mc_rows)
reject,p_adj,_,_=multipletests(mc.p_exact.values,alpha=.05,method='holm')
mc['p_holm']=p_adj
mc['reject_holm_0.05']=reject
mc.to_csv(TAB/'mcnemar_primary_holm.csv',index=False)

display(boot)
display(mc)


In [ ]:

# Aggregate bootstrap deltas across the three seeds for compact reporting.
# This is a descriptive across-seed summary of the seed-specific paired estimates.
boot_agg=boot.groupby(['model_a','model_b']).agg(
    mean_delta_macro_f1=('delta_macro_f1_a_minus_b','mean'),
    min_ci_low=('ci95_low','min'),
    max_ci_high=('ci95_high','max'),
    n_seeds_ci_excludes_zero=('ci_excludes_zero','sum'),
    n_seeds_within_margin_0_01=('ci_entirely_within_practical_margin_0.01','sum'),
).reset_index()
boot_agg.to_csv(TAB/'paired_bootstrap_primary_summary_across_seeds.csv',index=False)
display(boot_agg)


In [ ]:

# Publication figures
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

plot=summary.reset_index()
plt.figure(figsize=(10,5.8))
plt.errorbar(np.arange(len(plot)),plot.mean_macro_f1,yerr=plot.sd_macro_f1,fmt='o',capsize=4)
plt.xticks(np.arange(len(plot)),plot.model,rotation=55,ha='right')
plt.ylabel('Macro-F1')
plt.title('Model comparison across repeated outer-fold evaluation')
plt.tight_layout()
plt.savefig(FIG/'all_models_macro_f1.png',dpi=300,bbox_inches='tight')
plt.show()

# Runtime figure for models that recorded fit_seconds
if 'fit_seconds' in metrics.columns:
    rt=metrics.dropna(subset=['fit_seconds']).groupby('model').fit_seconds.mean().sort_values()
    if len(rt):
        plt.figure(figsize=(8,5))
        plt.barh(rt.index,rt.values)
        plt.xlabel('Mean fit/search time per outer fold (s)')
        plt.title('Computational cost for recorded pipelines')
        plt.tight_layout()
        plt.savefig(FIG/'model_fit_time.png',dpi=300,bbox_inches='tight')
        plt.show()

# Mean normalized confusion matrices across seeds for top 3 models.
top3=summary.head(3).index.tolist()
labels=classes
for model in top3:
    cms=[]
    for seed in SEEDS:
        g=preds[(preds.model==model)&(preds.seed==seed)].drop_duplicates('row_id')
        cm=confusion_matrix(g.y_true.astype(str),g.y_pred.astype(str),labels=labels,normalize='true')
        cms.append(cm)
    cm_mean=np.mean(cms,axis=0)
    pd.DataFrame(cm_mean,index=labels,columns=labels).to_csv(TAB/f'confusion_normalized_mean_{model}.csv')
    fig,ax=plt.subplots(figsize=(6,5))
    im=ax.imshow(cm_mean,vmin=0,vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=45,ha='right')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Mean normalized OOF confusion — {model}')
    for i in range(cm_mean.shape[0]):
        for j in range(cm_mean.shape[1]):
            ax.text(j,i,f'{cm_mean[i,j]:.3f}',ha='center',va='center')
    fig.colorbar(im,ax=ax)
    plt.tight_layout()
    plt.savefig(FIG/f'confusion_normalized_mean_{model}.png',dpi=300,bbox_inches='tight')
    plt.close()
print('Top 3:', top3)


In [ ]:

# Descriptive LDA structure for interpretation only (not used for unbiased predictive performance estimation)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

df=pd.read_excel(DATA/'INIAP_Dataset.xlsx').rename(columns={'AspectRation':'AspectRatio'})
X=df.drop(columns=[TARGET]); y=df[TARGET].astype(str)
features=X.columns.tolist()
Xs=StandardScaler().fit_transform(X)
le=LabelEncoder(); yy=le.fit_transform(y)
lda=LinearDiscriminantAnalysis(n_components=3).fit(Xs,yy)
lda_scalings=pd.DataFrame(lda.scalings_[:,:3],index=features,columns=['LD1','LD2','LD3'])
lda_scalings.to_csv(TAB/'lda_scalings_descriptive_full_dataset.csv')
# absolute coefficients for easier ranking
abs_rank=lda_scalings.abs().rank(ascending=False,method='min')
abs_rank.to_csv(TAB/'lda_scalings_absolute_rank.csv')
display(lda_scalings)
print('NOTE: full-dataset LDA structure is descriptive/interpretive only; predictive performance comes exclusively from nested/outer-fold pipelines.')


In [ ]:

run_info={
    'seeds':SEEDS,
    'n_models':int(metrics.model.nunique()),
    'primary_comparisons':[list(x) for x in PRIMARY_COMPARISONS],
    'bootstrap_resamples_per_seed':B,
    'mcnemar_multiple_testing_correction':'Holm across all pre-specified seed-level tests',
    'practical_difference_margin_macro_f1':0.01,
    'equivalence_language':'CI-within-margin is reported as practical-equivalence evidence, not formal TOST equivalence.',
    'confusion_matrix_note':'Mean of per-seed row-normalized OOF confusion matrices; avoids treating repeated-seed predictions as independent grains.',
}
with open(LOG/'NB06_run_info.json','w') as f:
    json.dump(run_info,f,indent=2)
print('NB06 FIXED completed successfully.')
